# Fase 2 — Sensibilità alle dipendenze degli episodi OOD

Esegui SOLO DOPO PyCharm (`test_ood_dependency_sensitivity.py`). Questo modulo di audit separato legge ground truth per identificare **possibili** dipendenze tra fault record dello stesso cloudbed, stessa classe e stesso componente, considerando distanze 0, 30, 60, 120 e 240 minuti. Raggruppa anche tutte le finestre sovrapposte. L'output è **solo aggregato**, senza case ID o etichette per caso. La vicinanza/identità dei campi è un indicatore di rischio, non una prova di dipendenza fisica; la loro assenza NON dimostra indipendenza.

**Nessuna telemetria, nessuna inferenza, nessuna scrittura, nessuna selezione discovery/test.**


In [ ]:
# FASE 2 — analisi di sensibilità della dipendenza tra eventi OOD.
# Ground truth LETTO solo da modulo di VALUTAZIONE (non dall'agente).
# NON prova l'indipendenza fisica; emette SOLO aggregati e ZERO API LLM.
from google.colab import drive
from pathlib import Path
import json
import os
import re
import subprocess
import sys

REPO = Path("/content/Tesi_OpenRCA_OOD_Audit")
DATA = Path("/content/drive/MyDrive/Tesi_OpenRCA")
BRANCH = "feature/ood-knowledge-transfer-audit-2026-11-10"
URL = "https://github.com/alessio1996A/Tesi_Laurea_Magistrale_OpenRCA.git"

drive.mount("/content/drive")
for file in (
    DATA / "Market.zip",
    DATA / "data/processed/split_market_v1.csv",
    DATA / "data/processed/benchmark_inputs_market.csv",
):
    if not file.is_file():
        raise FileNotFoundError("Input mancante: " + file.name)

if REPO.exists():
    if not (REPO / ".git").is_dir():
        raise RuntimeError("Directory OOD non-Git: nessuna sovrascrittura")
    actual = subprocess.check_output(
        ["git", "-C", str(REPO), "branch", "--show-current"], text=True
    ).strip()
    if actual != BRANCH:
        raise RuntimeError("Branch diverso: non eseguo reset/checkout automatico")
    if subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"], text=True
    ).strip():
        raise RuntimeError("Modifiche locali: non eseguo pull")
    subprocess.run(
        ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
        check=True, text=True, capture_output=True
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--single-branch", "--branch",
         BRANCH, URL, str(REPO)],
        check=True, text=True, capture_output=True
    )
commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
print("=== FASE 2 — SENSIBILITÀ DIPENDENZA EPISODI OOD ===")
print("Branch:", BRANCH)
print("Commit:", commit)
env = dict(os.environ)
env["PYTHONPATH"] = str(REPO / "src")
r = subprocess.run(
    [sys.executable, "-m", "evaluation.ood_dependency_sensitivity",
     "--data-dir", str(DATA)],
    cwd=str(REPO), env=env, text=True, capture_output=True
)
if r.returncode:
    raise RuntimeError("Audit fallito: nessuna inferenza eseguita.\n"
                       + r.stderr[-5000:])
try:
    data = json.loads(r.stdout)
except json.JSONDecodeError as exc:
    raise RuntimeError("Audit non restituisce JSON validabile") from exc
if (
    data.get("audit") != "ood_dependency_sensitivity_controlled_gt_v1"
    or data.get("source_verified") is not True
    or data.get("ood_query_count") != 15
    or data.get("shared_window_pairs") != 1
    or data.get("read_ground_truth") is not True
    or data.get("read_telemetry") is not False
    or data.get("emits_case_label_mapping") is not False
    or data.get("files_written") is not False
    or data.get("model_api_calls") != 0
):
    raise RuntimeError("Risultato inatteso: report non validato")
dump = json.dumps(data, indent=2, ensure_ascii=False)
if re.search(r"cloudbed-[12]_q\d+", dump):
    raise RuntimeError("Output potenzialmente contiene mappature caso/etichetta")
print("\n=== CONTEGGI AGGREGATI PER SOGLIA (NON PROVA DI INDIPENDENZA) ===")
print(dump)
print("\nZero API. Nessuna selezione discovery/test, nessun file scritto.")


## Dopo l'esecuzione

Condividi il JSON completo. Dopo questo controllo decideremo la fattibilità della separazione discovery/test e come riportare i limiti statistici nella tesi.
